# adsb.lol API — Exploration

Direct HTTP access, no auth, no API key.  
Base URL: `https://api.adsb.lol/v2/`  

**Note:** The docs reference `/api/v2/` — in practice only `/v2/` works (tested 2026-05-18).

In [ ]:
import requests
import json
import time
import pandas as pd
from datetime import datetime

BASE_URL = "https://api.adsb.lol/v2"

def get(path, **kwargs):
    url = f"{BASE_URL}{path}"
    r = requests.get(url, timeout=10, **kwargs)
    r.raise_for_status()
    return r.json()

print("Setup OK")

## 1. Live aircraft over Berlin (25 nm radius)

In [ ]:
data = get("/lat/52.52/lon/13.41/dist/25")

print(f"Timestamp:   {datetime.fromtimestamp(data['now'] / 1000)}")
print(f"Aircraft:    {data['total']}")
print(f"Server time: {data['ptime']} ms")

aircraft = data["ac"]
print(f"\nFirst 3 entries (raw):")
for a in aircraft[:3]:
    print(json.dumps(a, indent=2))

## 2. Field analysis — what is always present, what is sometimes missing?

In [ ]:
df = pd.DataFrame(aircraft)

print(f"Shape: {df.shape}")
print(f"\nSpalten:")
print(df.dtypes.to_string())

print(f"\nFehlende Werte pro Spalte:")
missing = df.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0].to_string())

In [ ]:
# Readable overview of all aircraft
cols = [c for c in ["hex", "flight", "r", "t", "alt_baro", "gs", "lat", "lon", "seen"] if c in df.columns]
print(df[cols].to_string(index=False))

## 3. Callsign search — all Lufthansa flights worldwide

In [ ]:
lh = get("/callsign/DLH")

print(f"Lufthansa flights currently airborne: {lh['total']}")

df_lh = pd.DataFrame(lh["ac"])
cols = [c for c in ["hex", "flight", "r", "t", "alt_baro", "gs", "lat", "lon"] if c in df_lh.columns]
print(df_lh[cols].head(20).to_string(index=False))

## 4. Single aircraft by ICAO24 hex

In [ ]:
# Take first hex from the Berlin query
sample_hex = aircraft[0]["hex"]
print(f"Looking up: {sample_hex}")

single = get(f"/hex/{sample_hex}")
print(json.dumps(single["ac"], indent=2) if single["ac"] else "No longer visible")

## 5. Data quality — alt_baro: number vs. 'ground'

In [ ]:
# alt_baro can be int OR string 'ground' — an ETL edge case
if "alt_baro" in df.columns:
    ground = df[df["alt_baro"] == "ground"]
    airborne = df[df["alt_baro"] != "ground"]
    print(f"On ground:  {len(ground)}")
    print(f"Airborne:   {len(airborne)}")

    # ETL normalisation: clean types
    df["alt_baro_ft"] = pd.to_numeric(df["alt_baro"], errors="coerce")  # 'ground' → NaN
    df["on_ground"] = df["alt_baro"] == "ground"
    print(f"\nAfter normalisation:")
    print(df[["hex", "alt_baro", "alt_baro_ft", "on_ground"]].to_string(index=False))

## 6. Measure response time — how fast is the API?

In [ ]:
timings = []
for i in range(5):
    t0 = time.time()
    d = get("/lat/52.52/lon/13.41/dist/25")
    elapsed = (time.time() - t0) * 1000
    timings.append(elapsed)
    print(f"Request {i+1}: {elapsed:.0f} ms, {d['total']} aircraft")
    time.sleep(2)

print(f"\nAverage: {sum(timings)/len(timings):.0f} ms")
print(f"Min/Max: {min(timings):.0f} / {max(timings):.0f} ms")

## 7. Comparison: adsb.lol vs. OpenSky — overlap check

Which callsigns do both sources see over Berlin right now?

In [ ]:
import sys
sys.path.append('.')

# adsb.lol callsigns over Berlin
adsb_callsigns = set()
for a in aircraft:
    cs = a.get("flight", "").strip()
    if cs:
        adsb_callsigns.add(cs)

print(f"adsb.lol callsigns over Berlin: {sorted(adsb_callsigns)}")

# OpenSky for comparison (live state vectors over Berlin bounding box)
try:
    from opensky_api.client import OpenSkyClient
    os_client = OpenSkyClient(use_mock=False)
    states = os_client.get_states(bbox=(51.9, 52.9, 12.9, 14.0))
    os_callsigns = {s.get("callsign", "").strip() for s in states if s.get("callsign")}
    print(f"OpenSky callsigns over Berlin:  {sorted(os_callsigns)}")
    overlap = adsb_callsigns & os_callsigns
    print(f"\nBoth see:      {sorted(overlap)}")
    print(f"Only adsb.lol: {sorted(adsb_callsigns - os_callsigns)}")
    print(f"Only OpenSky:  {sorted(os_callsigns - adsb_callsigns)}")
except Exception as e:
    print(f"OpenSky not available: {e}")
    print("(Overlap check requires active credentials)")

## 8. Raw JSON for MongoDB — what does a landing zone document look like?

In [ ]:
from datetime import timezone

# Exactly how the collector stores the document in MongoDB
landing_doc = {
    "collected_at": datetime.now(timezone.utc).isoformat(),
    "query_type": "lat_lon_dist",
    "query_params": {"lat": 52.52, "lon": 13.41, "dist": 25},
    "source": "adsb.lol",
    "total": data["total"],
    "now": data["now"],
    "ac": data["ac"],
}

print(f"Document size:    {len(json.dumps(landing_doc))} bytes")
print(f"Top-level fields: {list(landing_doc.keys())}")
print(f"Aircraft entries: {len(landing_doc['ac'])}")
print(f"\nEstimated MongoDB throughput at 60s polling: ~{len(json.dumps(landing_doc)) * 60 // 1024} KB/min")